# CIFAR-100-LT: Five ResNet-18 experiments with one held-out test set

Run this notebook from top to bottom. First split the labeled data once into 80% training and 20% testing, stratified across all 100 classes. The test split is stored separately and never used for training, OOF rejection, diffusion, generation, class weighting or model selection. All five final ResNet-18 checkpoints must be complete and frozen before test images are opened for final evaluation.

| Experiment | Training data |
| --- | --- |
| original_real_only | All original training images |
| original_overlap_rejected | Training images retained by the original OOF overlap/confidence rejection |
| diffusion_real_only_control | All original training images, an independent baseline repeat |
| synthetic_only | Class-conditional diffusion-generated images only |
| real_plus_synthetic | All original training images plus generated images |

The original three diffusion comparisons included a real-only baseline, which overlaps with the first original experiment. To preserve exactly the requested five runs, the third row is a separately trained baseline replicate (different seed), not a different data treatment. Set REPEAT_MODE = 'filtered_plus_synthetic' before running if you prefer five distinct data treatments instead; the third row then uses filtered real images plus synthetic images. The other four runs use the same initialization seed. The original historical notebook is retained unchanged.

The split uses a new fixed split seed (2026), and all models start from scratch in a new output directory. Earlier experiments used this same underlying dataset, so this is a fresh within-dataset holdout for the new pipeline, not a fully independent external benchmark. Do not use the final test scores to select hyperparameters.

## 1. Setup

Activate a GPU-enabled Python environment and install torch/torchvision and:
```bash
python -m pip install kagglehub numpy pandas pillow matplotlib scikit-learn tqdm jupyter ipykernel
```
A CUDA GPU is strongly recommended. Kaggle competition access is required. Checkpoints and generated images are saved locally. To smoke-test the pipeline, set FOLD_EPOCHS=1, FINAL_EPOCHS=1, DIFFUSION_EPOCHS=1, IMAGES_PER_CLASS=2, SAMPLE_STEPS=4 and use a separate SAVE_DIR; smoke-test scores are not scientific results.

In [ ]:
import os
import gc
import json
import random
from pathlib import Path
from copy import deepcopy
from contextlib import nullcontext
import math
import hashlib
from torch.utils.data import WeightedRandomSampler

import kagglehub

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_recall_fscore_support, roc_auc_score, average_precision_score
)

# ----- Edit these settings -----
SAVE_DIR = Path('./cifar100_lt_five_experiments_runs').resolve()
SEED = 42                      # Model / fold / generation seed
SPLIT_SEED = 2026               # Fixed new held-out test split
TEST_SIZE = 0.20
REPEAT_MODE = 'baseline_repeat' # or 'filtered_plus_synthetic'
FINAL_EPOCHS = 15
BATCH_SIZE = 128
NUM_WORKERS = 0                 # Safest for notebooks/Windows; try 2-4 for speed
LEARNING_RATES = {'cnn': 0.001, 'resnet18': 0.0005}
FOLDS = 5
MODELS_EACH = 5
DRAWS_PER_MODEL = 4
FOLD_EPOCHS = 12
TRUE_PROB_THRESHOLD = 0.05
MAX_REJECT_PER_CLASS = 0.10
MIN_KEEP_PER_CLASS = 5
assert REPEAT_MODE in ('baseline_repeat', 'filtered_plus_synthetic')
assert FOLD_EPOCHS > 0

DIFFUSION_EPOCHS = 100
DIFFUSION_BATCH_SIZE = 64
DIFFUSION_LR = 2e-4
DIFFUSION_WIDTH = 48
DIFFUSION_TIMESTEPS = 1000
EMA_DECAY = 0.999
BALANCED_DIFFUSION = True
IMAGES_PER_CLASS = 200
SAMPLE_STEPS = 50                # Deterministic DDIM reverse steps
SAMPLE_BATCH_SIZE = 32
assert DIFFUSION_EPOCHS > 0 and FINAL_EPOCHS > 0
assert IMAGES_PER_CLASS > 0 and SAMPLE_BATCH_SIZE > 0
assert 2 <= SAMPLE_STEPS <= DIFFUSION_TIMESTEPS
assert DIFFUSION_WIDTH % 8 == 0

# ----- GPU selection -----
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

print('Device:', DEVICE)
if DEVICE.type == 'cpu':
    print('Warning: CPU training is much slower. Install GPU-enabled PyTorch if possible.')


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
SAVE_DIR.mkdir(parents=True, exist_ok=True)
(SAVE_DIR / 'checkpoints').mkdir(exist_ok=True)
(SAVE_DIR / 'generated').mkdir(exist_ok=True)
(SAVE_DIR / 'results').mkdir(exist_ok=True)
(SAVE_DIR / 'splits').mkdir(exist_ok=True)
(SAVE_DIR / 'oof').mkdir(exist_ok=True)

## 2. Split the labeled dataset before any training

The Kaggle competition's official test images are unlabeled and are excluded. Create fixed absolute-path manifests for the labeled train and held-out test partitions. The held-out dataframe is deleted from memory after writing the manifest, and the test images are only loaded in the final evaluation cell. The split is manifest-based rather than a physical copy of the files. Both splits must contain all classes.

In [ ]:
kagglehub.login()

In [ ]:
# Download CIFAR-100-LT competition data.
# Authenticate with Kaggle in the previous cell if required.

folder = Path(kagglehub.competition_download(
    'cifar100-image-classification-with-long-tail'
))
print('KaggleHub dataset folder:', folder)

# KaggleHub may return a folder with or without a "dataset" subfolder.
data_root = None
for root in [folder / 'dataset', folder]:
    csv_file = root / 'cifar100_lt' / 'data' / 'cifar100_lt.csv'
    if csv_file.exists():
        data_root = root
        csv_path = csv_file
        break

if data_root is None:
    raise FileNotFoundError(
        f'cifar100_lt.csv was not found under {folder}. '
        'Check the downloaded competition files.'
    )

print('CSV path:', csv_path)

all_rows = pd.read_csv(csv_path, header=None, names=['image_path', 'label'],
                       dtype={'label': 'string'})
paths = all_rows['image_path'].str.replace(chr(92), '/', regex=False)
all_rows['image_path'] = paths
train_rows = all_rows[paths.str.contains('/train/', regex=False)].copy()
test_rows = all_rows[paths.str.contains('/test/', regex=False)].copy()

if train_rows.empty or train_rows['label'].isna().any():
    raise ValueError('Training images/labels were not found in the CSV.')

class_names = sorted(train_rows['label'].unique())
class_to_id = {label: i for i, label in enumerate(class_names)}
NUM_CLASSES = len(class_names)
if NUM_CLASSES != 100:
    raise ValueError(f'Expected 100 classes, found {NUM_CLASSES}.')
train_rows['target'] = train_rows['label'].map(class_to_id).astype(int)

train_rows['image_path'] = train_rows['image_path'].map(lambda p: str((data_root / p).resolve()))
assert train_rows['image_path'].is_unique
train_pool, heldout_test = train_test_split(
    train_rows, test_size=TEST_SIZE,
    random_state=SPLIT_SEED, stratify=train_rows['target']
)
train_pool = train_pool.reset_index(drop=True)
heldout_test = heldout_test.reset_index(drop=True)
test_rows = test_rows.reset_index(drop=True)
print('Train pool:', len(train_pool), '| Held-out test:', len(heldout_test), '| Kaggle test:', len(test_rows))

# Persist fixed split manifests before any training.
TRAIN_MANIFEST = SAVE_DIR / 'splits' / 'train.csv'
TEST_MANIFEST = SAVE_DIR / 'splits' / 'heldout_test.csv'

def persist_split(table, path):
    payload = table.to_csv(index=False).encode('utf-8')
    if path.exists() and path.read_bytes() != payload:
        raise RuntimeError(f'Split changed: {path}. Use a new SAVE_DIR.')
    if not path.exists():
        temp = path.with_suffix('.tmp')
        temp.write_bytes(payload)
        os.replace(temp, path)
    return hashlib.sha256(payload).hexdigest()

assert set(train_pool['image_path']).isdisjoint(heldout_test['image_path'])
assert len(train_pool) + len(heldout_test) == len(train_rows)
assert train_pool['target'].nunique() == heldout_test['target'].nunique() == NUM_CLASSES
train_split_sha = persist_split(train_pool, TRAIN_MANIFEST)
test_split_sha = persist_split(heldout_test, TEST_MANIFEST)
del heldout_test, train_rows, test_rows, all_rows, paths

settings = {
    'experiment_version': 'five_experiments_holdout_v1',
    'dataset_hash': hashlib.sha256(csv_path.read_bytes()).hexdigest(),
    'dataset_folder': str(data_root.resolve()),
    'seed': SEED, 'split_seed': SPLIT_SEED, 'test_size': TEST_SIZE,
    'repeat_mode': REPEAT_MODE, 'folds': FOLDS, 'models_each': MODELS_EACH,
    'draws_per_model': DRAWS_PER_MODEL, 'fold_epochs': FOLD_EPOCHS,
    'true_prob_threshold': TRUE_PROB_THRESHOLD,
    'max_reject_per_class': MAX_REJECT_PER_CLASS,
    'min_keep_per_class': MIN_KEEP_PER_CLASS, 'num_workers': NUM_WORKERS,
    'train_split_sha256': train_split_sha, 'test_split_sha256': test_split_sha,
    'final_epochs': FINAL_EPOCHS, 'batch_size': BATCH_SIZE,
    'learning_rates': LEARNING_RATES, 'diffusion_epochs': DIFFUSION_EPOCHS,
    'diffusion_batch_size': DIFFUSION_BATCH_SIZE, 'diffusion_lr': DIFFUSION_LR,
    'width': DIFFUSION_WIDTH, 'timesteps': DIFFUSION_TIMESTEPS,
    'ema_decay': EMA_DECAY, 'balanced_diffusion': BALANCED_DIFFUSION,
    'images_per_class': IMAGES_PER_CLASS, 'sample_steps': SAMPLE_STEPS,
    'sample_batch_size': SAMPLE_BATCH_SIZE,
}
settings_file = SAVE_DIR / 'settings.json'
if settings_file.exists():
    old_settings = json.loads(settings_file.read_text())
    if settings != old_settings:
        raise ValueError('Settings changed! Set SAVE_DIR to a NEW folder for this experiment.')
else:
    settings_file.write_text(json.dumps(settings, indent=2))

print('Training and held-out test manifests saved separately.')
# All subsequent training/OOF inputs must be sourced from this fixed pool.
real_training_paths = frozenset(train_pool['image_path'])
assert train_pool.groupby('target').size().min() >= FOLDS

## 3. Original class distribution

The diffusion training loader can draw real examples with inverse-frequency sampling so each class is equally likely. This increases exposure to rare classes but may also increase memorization; it does not create new real information.

In [ ]:
counts = train_pool['label'].value_counts().sort_values()
print('Smallest class:', counts.min(), 'images | Largest class:', counts.max(), 'images')
plt.figure(figsize=(13, 15))
plt.barh([str(x) for x in counts.index], counts.values)
plt.xlabel('Training images')
plt.title('CIFAR-100-LT class counts')
plt.tight_layout()
plt.show()

## 4. Classifier definitions and checkpointed training

Reuse the original CNN and CIFAR-sized ResNet-18, with the original fixed normalization, augmentation, AdamW and inverse-square-root weighted cross-entropy. The CNN is used only for the overlap experiment's OOF predictions. No classifier sees test images during fitting. Training inputs must belong to the fixed training split or the generated-image manifest. Each checkpoint records the training-data fingerprint, seed, architecture and epoch budget to reject accidental cross-run reuse.

In [ ]:
mean = (0.5071, 0.4867, 0.4408)
std = (0.2675, 0.2565, 0.2761)
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])


class CIFARDataset(Dataset):
    def __init__(self, table, transform):
        self.paths = [data_root / path for path in table['image_path']]
        self.labels = table['target'].tolist() if 'target' in table else [-1] * len(table)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            image = image.convert('RGB')
        return self.transform(image), self.labels[index]


class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, NUM_CLASSES),
        )

    def forward(self, x):
        return self.layers(x)


def make_model(kind):
    if kind == 'cnn':
        return SimpleCNN().to(DEVICE)
    model = models.resnet18(weights=None)
    model.conv1 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model.to(DEVICE)


def make_loader(table, training=False):
    transform = train_transform if training else test_transform
    return DataLoader(
        CIFARDataset(table, transform), batch_size=BATCH_SIZE,
        shuffle=training, num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == 'cuda')
    )


def class_weights(table):
    counts = np.bincount(table['target'], minlength=NUM_CLASSES)
    weights = np.zeros(NUM_CLASSES, dtype=float)
    present = counts > 0
    weights[present] = 1 / np.sqrt(counts[present])
    weights[present] /= weights[present].mean()
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)

In [ ]:
def assert_training_sources(data):
    paths = set(data['image_path'].map(lambda p: str((data_root / p).resolve())))
    allowed = real_training_paths
    if 'synthetic_pool' in globals():
        allowed = allowed.union(synthetic_pool['image_path'])
    if not paths.issubset(allowed):
        raise RuntimeError('Training input includes a path outside the authorized training pools.')

def train_model(kind, name, data, epochs, seed):
    assert_training_sources(data)
    signature = {
        'data_sha256': hashlib.sha256(
            data[['image_path', 'label', 'target']].to_csv(index=False).encode('utf-8')
        ).hexdigest(),
        'kind': kind, 'seed': seed, 'epochs': epochs,
        'settings_sha256': hashlib.sha256((SAVE_DIR / 'settings.json').read_bytes()).hexdigest(),
    }
    set_seed(seed)
    model = make_model(kind)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATES[kind], weight_decay=0.01)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights(data))
    loader = make_loader(data, training=True)
    checkpoint = SAVE_DIR / 'checkpoints' / f'{name}.pt'
    history = []
    start_epoch = 0

    if checkpoint.exists():
        saved = torch.load(checkpoint, map_location=DEVICE, weights_only=True)
        if saved.get('signature') != signature:
            raise RuntimeError(f'Checkpoint training inputs changed: {name}. Use a new SAVE_DIR.')
        model.load_state_dict(saved['model'])
        optimizer.load_state_dict(saved['optimizer'])
        history = saved['history']
        start_epoch = saved['epoch']
        del saved
        if not 0 <= start_epoch <= epochs:
            raise RuntimeError(f'Invalid checkpoint epoch: {name}')
        print(f'{name}: resuming at epoch {start_epoch}/{epochs}')

    for epoch in range(start_epoch, epochs):
        # Fixed per-epoch seed also makes checkpoint restarts reproducible.
        set_seed(seed + epoch + 1)
        model.train()
        total_loss = 0.0
        correct = 0
        seen = 0

        for images, labels in tqdm(loader, desc=f'{name}: epoch {epoch+1}/{epochs}', leave=False):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(labels)
            correct += (outputs.argmax(1) == labels).sum().item()
            seen += len(labels)

        row = {'epoch': epoch + 1, 'loss': total_loss / seen, 'accuracy': correct / seen}
        history.append(row)
        print(f"{name}: epoch {epoch+1}/{epochs} | loss {row['loss']:.4f} | accuracy {row['accuracy']:.4f}")

        # Save without risking an incomplete final checkpoint.
        temp_path = checkpoint.with_suffix('.tmp')
        torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                    'epoch': epoch + 1, 'history': history, 'signature': signature}, temp_path)
        os.replace(temp_path, checkpoint)

    pd.DataFrame(history).to_csv(SAVE_DIR / 'results' / f'{name}_training.csv', index=False)
    return model


@torch.inference_mode()
def predict_probs(model, data):
    if len(data) == 0:
        return np.empty((0, NUM_CLASSES), dtype=np.float32)
    model.eval()
    predictions = []
    for images, _ in make_loader(data):
        outputs = model(images.to(DEVICE))
        predictions.append(outputs.softmax(dim=1).cpu().numpy())
    return np.concatenate(predictions)


def free_model(model):
    model.to('cpu')  # Release the model's GPU weights before clearing the cache.
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    elif DEVICE.type == 'mps':
        torch.mps.empty_cache()

## 5. Original overlap rejection: five stratified folds

The **four fold numbers per model are sampled with replacement**. For example, `[1, 1, 3, 5]` means the model trains twice on fold 1 and once each on folds 3 and 5; it **never saw folds 2 and 4**.

We retry a random draw until every fold has at least one eligible CNN and one eligible ResNet, and each model sees at least three different folds. This is necessary to ensure that every image can receive two genuine out-of-fold predictions.

All folds are created exclusively from the 80% training partition. The held-out test set is not a sixth fold and is never used for OOF predictions.

In [ ]:
# Give each image one of five fold numbers (0, 1, 2, 3, or 4).
fold_id = np.full(len(train_pool), -1, dtype=int)
splitter = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
for fold, (_, held_out) in enumerate(splitter.split(train_pool, train_pool['target'])):
    fold_id[held_out] = fold
fold_rows = [np.where(fold_id == fold)[0] for fold in range(FOLDS)]

# Sample training folds WITH replacement, ensuring every held-out fold is covered.
rng = np.random.default_rng(SEED)
plans = {}
for kind in ['cnn', 'resnet18']:
    for _ in range(50000):
        draws = rng.integers(0, FOLDS, size=(MODELS_EACH, DRAWS_PER_MODEL))
        enough_variety = all(len(set(row)) >= 3 for row in draws)
        all_folds_covered = all(any(fold not in row for row in draws) for fold in range(FOLDS))
        if enough_variety and all_folds_covered:
            plans[kind] = draws
            break
    else:
        raise RuntimeError(f'Could not generate an eligible fold plan for {kind}.')

# Randomly choose one eligible CNN and one eligible ResNet FOR EACH IMAGE.
selected = {}
for kind in ['cnn', 'resnet18']:
    selected[kind] = np.full(len(train_pool), -1, dtype=int)
    for fold in range(FOLDS):
        eligible = [m for m in range(MODELS_EACH) if fold not in plans[kind][m]]
        selected[kind][fold_rows[fold]] = rng.choice(eligible, size=len(fold_rows[fold]))

    # A selected model must NEVER train on the image's fold.
    for m in range(MODELS_EACH):
        image_folds = fold_id[selected[kind] == m]
        assert not np.isin(image_folds, plans[kind][m]).any(), 'Fold leakage detected!'

for kind, draws in plans.items():
    for m, row in enumerate(draws):
        print(f'{kind} {m+1}: training folds {row.tolist()}')

(SAVE_DIR / 'results' / 'fold_sampling_plan.json').write_text(
    json.dumps({kind: draws.tolist() for kind, draws in plans.items()}, indent=2)
)
assignments = train_pool[['image_path', 'label']].copy()
assignments['fold'] = fold_id + 1
assignments['chosen_cnn'] = selected['cnn'] + 1
assignments['chosen_resnet18'] = selected['resnet18'] + 1
assignments.to_csv(SAVE_DIR / 'results' / 'image_model_assignments.csv', index=False)
print('All images have two valid out-of-fold model assignments.')

## 6. Train the ten OOF models and collect training-only predictions

The model only predicts for the images **assigned to it**. Predictions are cached to disk, so a rerun can skip completed models. A cache entry records its image indices, labels, and training folds for safety.

The same five CNNs and five ResNet-18s, fold bootstrapping with replacement, 12-epoch defaults and assignment policy are retained. Only predictions on training-pool rows omitted from that model's training folds are collected.

In [ ]:
oof_probs = {
    'cnn': np.full((len(train_pool), NUM_CLASSES), np.nan, dtype=np.float32),
    'resnet18': np.full((len(train_pool), NUM_CLASSES), np.nan, dtype=np.float32),
}

for kind in ['cnn', 'resnet18']:
    for m in range(MODELS_EACH):
        draws = plans[kind][m]
        name = f'fold_{kind}_{m+1}'
        # Concatenation repeats a fold if it was sampled twice.
        training_indices = np.concatenate([fold_rows[f] for f in draws])
        prediction_indices = np.where(selected[kind] == m)[0]
        assert not np.isin(fold_id[prediction_indices], draws).any()

        training_data = train_pool.iloc[training_indices].reset_index(drop=True)
        prediction_data = train_pool.iloc[prediction_indices].reset_index(drop=True)
        cache = SAVE_DIR / 'oof' / f'{name}.npz'

        if cache.exists():
            with np.load(cache) as saved:
                assert np.array_equal(saved['indices'], prediction_indices)
                assert np.array_equal(saved['labels'], prediction_data['target'].to_numpy())
                assert np.array_equal(saved['draws'], draws)
                assert np.array_equal(saved['paths'], prediction_data['image_path'].to_numpy(dtype=str))
                probs = saved['probs']
            print(name, ': cached predictions loaded')
        else:
            print(f'{name}: train {len(training_data)} rows, predict {len(prediction_data)} held-out rows')
            model = train_model(kind, name, training_data, FOLD_EPOCHS,
                                seed=SEED + (100 if kind == 'cnn' else 200) + m)
            probs = predict_probs(model, prediction_data)
            # Atomic NPZ save, so interrupted writes cannot damage the old cache.
            temp_cache = cache.with_suffix('.tmp.npz')
            np.savez_compressed(temp_cache, indices=prediction_indices,
                                labels=prediction_data['target'].to_numpy(),
                                draws=draws, probs=probs, paths=prediction_data['image_path'].to_numpy(dtype=str))
            os.replace(temp_cache, cache)
            free_model(model)
            del model

        assert probs.shape == (len(prediction_indices), NUM_CLASSES)
        oof_probs[kind][prediction_indices] = probs

for kind in ['cnn', 'resnet18']:
    assert np.isfinite(oof_probs[kind]).all(), f'Missing OOF predictions for {kind}'
    assert np.allclose(oof_probs[kind].sum(axis=1), 1, atol=0.001)
print('Both out-of-fold probability tables are complete.')

## 7. Apply the original overlap rejection rule

An image is a rejection candidate when both assigned models predict the wrong class and each assigns a probability of 0.05 or less to the image's true class. Candidates are ranked within their class by low average true-class probability. At most 10% of examples from a class are removed, and at least five images from each class are retained. This produces the filtered training pool.

This procedure identifies examples that both models find difficult; it does not establish that the removed images have incorrect labels.

**Simple confidence rule:** reject a candidate only if both chosen models (a) predict the wrong class and (b) each assign the true label a probability of at most `TRUE_PROB_THRESHOLD`.

Even when many images satisfy that rule, reject no more than `MAX_REJECT_PER_CLASS` of each class and keep at least `MIN_KEEP_PER_CLASS`. This protects rare classes on the long-tailed dataset.

Only the original training pool is filtered. The diffusion model will still learn from the complete original training pool, not the filtered subset. No test images or test scores influence rejection.

In [ ]:
true_labels = train_pool['target'].to_numpy()
indices = np.arange(len(train_pool))
cnn_probs = oof_probs['cnn']
resnet_probs = oof_probs['resnet18']

cnn_pred = cnn_probs.argmax(axis=1)
resnet_pred = resnet_probs.argmax(axis=1)
cnn_true_prob = cnn_probs[indices, true_labels]
resnet_true_prob = resnet_probs[indices, true_labels]

both_wrong = (cnn_pred != true_labels) & (resnet_pred != true_labels)
low_confidence = (cnn_true_prob <= TRUE_PROB_THRESHOLD) & (
    resnet_true_prob <= TRUE_PROB_THRESHOLD)
candidates = both_wrong & low_confidence

# Sort eligible candidates by the two models' average probability of the true class.
risk = 1 - (cnn_true_prob + resnet_true_prob) / 2
rejected = np.zeros(len(train_pool), dtype=bool)
for label in range(NUM_CLASSES):
    class_rows = np.where(true_labels == label)[0]
    class_candidates = class_rows[candidates[class_rows]]
    max_remove = min(
        int(len(class_rows) * MAX_REJECT_PER_CLASS),
        len(class_rows) - MIN_KEEP_PER_CLASS
    )
    max_remove = max(0, max_remove)
    worst_first = class_candidates[np.argsort(-risk[class_candidates])]
    rejected[worst_first[:max_remove]] = True

filtered_pool = train_pool.loc[~rejected].reset_index(drop=True)
print('Both models wrong:', both_wrong.sum())
print('Confidence-based candidates:', candidates.sum())
print('Rejected:', rejected.sum())
print('Original:', len(train_pool), '| Filtered:', len(filtered_pool))

review = assignments.copy()
review['cnn_prediction'] = [class_names[i] for i in cnn_pred]
review['resnet_prediction'] = [class_names[i] for i in resnet_pred]
review['cnn_confidence'] = cnn_probs.max(axis=1)
review['resnet_confidence'] = resnet_probs.max(axis=1)
review['cnn_true_class_probability'] = cnn_true_prob
review['resnet_true_class_probability'] = resnet_true_prob
review['both_wrong'] = both_wrong
review['candidate'] = candidates
review['rejected'] = rejected
review.to_csv(SAVE_DIR / 'results' / 'oof_confidence_and_rejection.csv', index=False)
review.loc[rejected].to_csv(SAVE_DIR / 'results' / 'rejected_examples.csv', index=False)
filtered_pool.to_csv(SAVE_DIR / 'results' / 'filtered_training_pool.csv', index=False)

before = np.bincount(true_labels, minlength=NUM_CLASSES)
after = np.bincount(filtered_pool['target'], minlength=NUM_CLASSES)
rejection_by_class = pd.DataFrame({
    'class': class_names, 'original': before, 'filtered': after,
    'removed': before - after
})
rejection_by_class.to_csv(SAVE_DIR / 'results' / 'rejection_by_class.csv', index=False)
print(rejection_by_class.sort_values('removed', ascending=False).head(10).to_string(index=False))

plt.figure(figsize=(5, 4))
plt.bar(['Retained', 'Rejected'], [len(filtered_pool), int(rejected.sum())])
plt.title('Confidence-based filtering')
plt.ylabel('Images')
plt.tight_layout()
plt.show()
assert set(filtered_pool['image_path']).issubset(real_training_paths)
assert filtered_pool['target'].nunique() == NUM_CLASSES
assert len(filtered_pool) + int(rejected.sum()) == len(train_pool)
filtered_real_pool = filtered_pool.assign(source='real')

## 8. Conditional diffusion U-Net

The forward process samples Gaussian noise at a random time:
`x_t = sqrt(alpha_bar[t]) * x_0 + sqrt(1-alpha_bar[t]) * epsilon`.
The network predicts epsilon given the noisy image, time index, and integer class label. A sinusoidal time embedding and a learned class embedding condition every residual block. The objective is mean squared error between predicted and actual noise. Images are scaled to [-1, 1]; classifier normalization is applied separately later.

A cosine noise schedule and EMA of the model weights are used for sampling. The U-Net has skip connections at resolutions 32, 16 and 8.

In [ ]:
diffusion_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,)*3, (0.5,)*3),
])

class DiffusionDataset(Dataset):
    def __init__(self, table):
        self.paths = [data_root / p for p in table['image_path']]
        self.targets = table['target'].astype(int).tolist()

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as img:
            x = diffusion_transform(img.convert('RGB'))
        return x, self.targets[index]

def sinusoidal_time(t, dim):
    half = dim // 2
    frequencies = torch.exp(
        -math.log(10000) * torch.arange(half, device=t.device, dtype=torch.float32)
        / max(half - 1, 1)
    )
    angles = t.float()[:, None] * frequencies[None, :]
    return torch.cat([angles.sin(), angles.cos()], dim=1)

class ConditionalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, embedding_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.cond = nn.Linear(embedding_dim, 2 * out_channels)
        self.norm2 = nn.GroupNorm(8, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.skip = (nn.Identity() if in_channels == out_channels
                     else nn.Conv2d(in_channels, out_channels, 1))
        self.act = nn.SiLU()

    def forward(self, x, emb):
        h = self.conv1(self.act(self.norm1(x)))
        scale, shift = self.cond(self.act(emb)).chunk(2, dim=1)
        h = self.norm2(h) * (1 + scale[:, :, None, None]) + shift[:, :, None, None]
        return self.skip(x) + self.conv2(self.act(h))

class ConditionalUNet(nn.Module):
    def __init__(self, num_classes, width=48):
        super().__init__()
        w, e = width, 4 * width
        self.embedding_dim = e
        self.time_mlp = nn.Sequential(nn.Linear(e, e), nn.SiLU(), nn.Linear(e, e))
        self.class_embedding = nn.Embedding(num_classes, e)
        self.input_conv = nn.Conv2d(3, w, 3, padding=1)
        self.enc1 = ConditionalBlock(w, w, e)
        self.down1 = nn.Conv2d(w, 2*w, 4, stride=2, padding=1)
        self.enc2 = ConditionalBlock(2*w, 2*w, e)
        self.down2 = nn.Conv2d(2*w, 4*w, 4, stride=2, padding=1)
        self.enc3 = ConditionalBlock(4*w, 4*w, e)
        self.mid1 = ConditionalBlock(4*w, 4*w, e)
        self.mid2 = ConditionalBlock(4*w, 4*w, e)
        self.dec3 = ConditionalBlock(8*w, 4*w, e)
        self.up2 = nn.Sequential(nn.Upsample(scale_factor=2, mode='nearest'),
                                 nn.Conv2d(4*w, 2*w, 3, padding=1))
        self.dec2 = ConditionalBlock(4*w, 2*w, e)
        self.up1 = nn.Sequential(nn.Upsample(scale_factor=2, mode='nearest'),
                                 nn.Conv2d(2*w, w, 3, padding=1))
        self.dec1 = ConditionalBlock(2*w, w, e)
        self.output = nn.Sequential(nn.GroupNorm(8, w), nn.SiLU(),
                                    nn.Conv2d(w, 3, 3, padding=1))

    def forward(self, x, t, y):
        emb = self.time_mlp(sinusoidal_time(t, self.embedding_dim)) + self.class_embedding(y)
        h1 = self.enc1(self.input_conv(x), emb)
        h2 = self.enc2(self.down1(h1), emb)
        h3 = self.enc3(self.down2(h2), emb)
        h = self.mid2(self.mid1(h3, emb), emb)
        h = self.dec3(torch.cat([h, h3], dim=1), emb)
        h = self.dec2(torch.cat([self.up2(h), h2], dim=1), emb)
        h = self.dec1(torch.cat([self.up1(h), h1], dim=1), emb)
        return self.output(h)

def cosine_alpha_bar(steps, device):
    grid = torch.linspace(0, steps, steps + 1, dtype=torch.float64) / steps
    curve = torch.cos((grid + 0.008) / 1.008 * math.pi / 2).square()
    curve = curve / curve[0]
    beta = (1 - curve[1:] / curve[:-1]).clamp(1e-5, 0.999)
    return (1 - beta).cumprod(dim=0).float().to(device)

alpha_bar = cosine_alpha_bar(DIFFUSION_TIMESTEPS, DEVICE)

def add_noise(x0, t, noise):
    a = alpha_bar[t].view(-1, 1, 1, 1)
    return a.sqrt() * x0 + (1 - a).sqrt() * noise

# Lightweight shape, gradient and schedule check before expensive training.
set_seed(SEED)
check_model = ConditionalUNet(NUM_CLASSES, DIFFUSION_WIDTH).to(DEVICE)
x_check = torch.randn(2, 3, 32, 32, device=DEVICE)
t_check = torch.tensor([0, DIFFUSION_TIMESTEPS-1], device=DEVICE)
y_check = torch.tensor([0, NUM_CLASSES-1], device=DEVICE)
prediction = check_model(add_noise(x_check, t_check, x_check), t_check, y_check)
assert prediction.shape == x_check.shape and torch.isfinite(prediction).all()
prediction.square().mean().backward()
assert check_model.class_embedding.weight.grad is not None
assert (alpha_bar[1:] < alpha_bar[:-1]).all()
del check_model, prediction, x_check, t_check, y_check
print('Diffusion forward/backward and schedule checks passed.')

## 9. Train the diffusion model

Sample a time step and independent noise for each real image, then optimize the conditional noise predictor. Checkpoints include optimizer, EMA, AMP scaler and history. EMA is updated after successful optimizer steps. The training objective is denoising loss, not classification accuracy.

In [ ]:
def atomic_torch_save(payload, path):
    temp = path.with_suffix('.tmp')
    torch.save(payload, temp)
    os.replace(temp, path)

def diffusion_loader():
    dataset = DiffusionDataset(train_pool)
    sampler = None
    if BALANCED_DIFFUSION:
        counts = np.bincount(dataset.targets, minlength=NUM_CLASSES)
        assert (counts > 0).all()
        weights = torch.tensor(1.0 / counts[np.asarray(dataset.targets)], dtype=torch.double)
        sampler = WeightedRandomSampler(weights, len(dataset), replacement=True)
    return DataLoader(dataset, batch_size=DIFFUSION_BATCH_SIZE,
                      sampler=sampler, shuffle=sampler is None,
                      num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda')

def train_diffusion():
    set_seed(SEED + 500)
    model = ConditionalUNet(NUM_CLASSES, DIFFUSION_WIDTH).to(DEVICE)
    ema = deepcopy(model).eval().requires_grad_(False)
    optimizer = torch.optim.AdamW(model.parameters(), lr=DIFFUSION_LR, weight_decay=0.01)
    amp = DEVICE.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=amp)
    loader = diffusion_loader()
    path = SAVE_DIR / 'checkpoints' / 'conditional_diffusion.pt'
    history, start = [], 0
    if path.exists():
        saved = torch.load(path, map_location=DEVICE, weights_only=True)
        model.load_state_dict(saved['model'])
        ema.load_state_dict(saved['ema'])
        optimizer.load_state_dict(saved['optimizer'])
        scaler.load_state_dict(saved['scaler'])
        start, history = saved['epoch'], saved['history']
        del saved
        print(f'Resuming diffusion at epoch {start}/{DIFFUSION_EPOCHS}')

    for epoch in range(start, DIFFUSION_EPOCHS):
        set_seed(SEED + 501 + epoch)
        model.train()
        total, seen = 0.0, 0
        for x0, y in tqdm(loader, desc=f'Diffusion {epoch+1}/{DIFFUSION_EPOCHS}', leave=False):
            x0, y = x0.to(DEVICE), y.to(DEVICE)
            t = torch.randint(0, DIFFUSION_TIMESTEPS, (len(y),), device=DEVICE)
            noise = torch.randn_like(x0)
            xt = add_noise(x0, t, noise)
            optimizer.zero_grad(set_to_none=True)
            with (torch.autocast('cuda', dtype=torch.float16) if amp else nullcontext()):
                pred = model(xt, t, y)
                loss = nn.functional.mse_loss(pred.float(), noise)
            if not torch.isfinite(loss):
                raise FloatingPointError('Non-finite diffusion loss')
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            previous_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() >= previous_scale:
                with torch.no_grad():
                    for avg, current in zip(ema.parameters(), model.parameters()):
                        avg.lerp_(current.detach(), 1 - EMA_DECAY)
            total += loss.item() * len(y)
            seen += len(y)
        row = {'epoch': epoch+1, 'noise_mse': total / seen}
        history.append(row)
        print(f"Diffusion epoch {epoch+1}: MSE {row['noise_mse']:.5f}")
        atomic_torch_save({'model': model.state_dict(), 'ema': ema.state_dict(),
                           'optimizer': optimizer.state_dict(), 'scaler': scaler.state_dict(),
                           'epoch': epoch+1, 'history': history}, path)

    pd.DataFrame(history).to_csv(SAVE_DIR / 'results' / 'diffusion_training.csv', index=False)
    del model, optimizer
    return ema.eval()

diffusion_model = train_diffusion()
loss_history = pd.read_csv(SAVE_DIR / 'results' / 'diffusion_training.csv')
plt.plot(loss_history['epoch'], loss_history['noise_mse'])
plt.xlabel('Epoch')
plt.ylabel('Noise prediction MSE')
plt.title('Conditional diffusion training')
plt.show()

## 10. Generate new images separately for every class

Use deterministic DDIM reverse sampling (eta=0) from independent Gaussian noise with the class label fixed throughout each reverse trajectory. The default produces exactly 200 images per class, 20,000 in total. Class labels are the requested conditions, not verified human annotations; synthetic images may be poor or mislabeled visually.

PNG batches are reproducibly generated and a manifest records every image, class ID and label. Existing complete batches are reused only when the trained checkpoint and generation settings are unchanged. Inspect the contact sheet before interpreting classifier scores.

In [ ]:
@torch.inference_mode()
def sample_class(model, class_id, n):
    model.eval()
    x = torch.randn(n, 3, 32, 32, device=DEVICE)
    y = torch.full((n,), class_id, device=DEVICE, dtype=torch.long)
    steps = torch.linspace(DIFFUSION_TIMESTEPS-1, 0, SAMPLE_STEPS).round().long().tolist()
    for i, step in enumerate(steps):
        t = torch.full((n,), step, device=DEVICE, dtype=torch.long)
        a = alpha_bar[step]
        next_a = alpha_bar[steps[i+1]] if i+1 < len(steps) else a.new_tensor(1.0)
        noise = model(x, t, y)
        x0 = ((x - (1-a).sqrt() * noise) / a.sqrt()).clamp(-1, 1)
        # Recompute epsilon after clipping to preserve the DDIM decomposition.
        noise = (x - a.sqrt() * x0) / (1-a).sqrt()
        x = next_a.sqrt() * x0 + (1-next_a).sqrt() * noise
    if not torch.isfinite(x).all():
        raise FloatingPointError(f'Non-finite generated images for class {class_id}')
    return ((x.clamp(-1, 1) + 1) * 127.5).round().to(torch.uint8).cpu()

def generate_all_classes(model):
    # Guard against reusing images from different trained weights.
    checkpoint = SAVE_DIR / 'checkpoints' / 'conditional_diffusion.pt'
    with checkpoint.open('rb') as f:
        digest = hashlib.file_digest(f, 'sha256').hexdigest()
    fingerprint = SAVE_DIR / 'generated' / 'checkpoint_sha256.txt'
    if fingerprint.exists() and fingerprint.read_text().strip() != digest:
        raise RuntimeError('Diffusion checkpoint changed. Use a new SAVE_DIR; do not mix images.')
    if not fingerprint.exists():
        if any((SAVE_DIR / 'generated').glob('*/*.png')):
            raise RuntimeError('Untracked generated images exist. Use a new SAVE_DIR.')
        fingerprint.write_text(digest)

    records = []
    for label in tqdm(range(NUM_CLASSES), desc='Generating each class'):
        class_dir = SAVE_DIR / 'generated' / f'class_{label:03d}'
        class_dir.mkdir(exist_ok=True)
        for start in range(0, IMAGES_PER_CLASS, SAMPLE_BATCH_SIZE):
            stop = min(start + SAMPLE_BATCH_SIZE, IMAGES_PER_CLASS)
            paths = [class_dir / f'{i:05d}.png' for i in range(start, stop)]
            if not all(p.is_file() for p in paths):
                set_seed(SEED + 10000 + label * 100000 + start)
                batch = sample_class(model, label, len(paths))
                for pixels, path in zip(batch, paths):
                    temp = path.with_suffix('.tmp.png')
                    Image.fromarray(pixels.permute(1, 2, 0).numpy()).save(temp)
                    os.replace(temp, path)
            records.extend({'image_path': str(p.resolve()), 'label': class_names[label],
                            'target': label, 'source': 'synthetic'} for p in paths)
    table = pd.DataFrame(records)
    assert table['image_path'].is_unique
    assert len(table) == NUM_CLASSES * IMAGES_PER_CLASS
    assert (table.groupby('target').size().reindex(range(NUM_CLASSES)) == IMAGES_PER_CLASS).all()
    assert set(table['image_path']).isdisjoint(train_pool['image_path'])
    manifest = SAVE_DIR / 'results' / 'generated_manifest.csv'
    temp = manifest.with_suffix('.tmp')
    table.to_csv(temp, index=False)
    os.replace(temp, manifest)
    return table

synthetic_pool = generate_all_classes(diffusion_model)
print('Generated images:', len(synthetic_pool))
del diffusion_model
gc.collect()
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
elif DEVICE.type == 'mps':
    torch.mps.empty_cache()

real_pool = train_pool.assign(source='real')
augmented_pool = pd.concat([real_pool, synthetic_pool], ignore_index=True)
class_counts = pd.DataFrame({
    'class': class_names,
    'real': np.bincount(real_pool['target'], minlength=NUM_CLASSES),
    'synthetic': np.bincount(synthetic_pool['target'], minlength=NUM_CLASSES),
    'augmented': np.bincount(augmented_pool['target'], minlength=NUM_CLASSES)
})
class_counts.to_csv(SAVE_DIR / 'results' / 'diffusion_class_counts.csv', index=False)
print(class_counts.to_string(index=False))

In [ ]:
# Inspect one generated image for every class; labels are conditioning labels.
fig, axes = plt.subplots(10, 10, figsize=(18, 19))
for label, ax in enumerate(axes.flat):
    path = synthetic_pool.loc[synthetic_pool['target'] == label, 'image_path'].iloc[0]
    with Image.open(path) as img:
        ax.imshow(img)
    ax.set_title(str(class_names[label]).replace('_', ' '), fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.savefig(SAVE_DIR / 'results' / 'generated_contact_sheet.png', dpi=130)
plt.show()

# Compare real vs generated counts per class.
ax = class_counts.sort_values('real').set_index('class')[['real', 'synthetic']].plot(
    kind='bar', figsize=(20, 5))
ax.set_ylabel('Images')
plt.tight_layout()
plt.show()

## 11. Train and freeze all five final ResNet-18 models

Do not evaluate any of these models on held-out test images yet. All checkpoints use fixed epochs and their assigned data manifests. The original baseline, filtered baseline, synthetic-only and augmented models share the same initialization seed; the real-only repeat deliberately uses a different seed. When REPEAT_MODE is filtered_plus_synthetic, all five models use the same initialization seed.

Equal epochs do not imply equal optimizer-update counts. The training set sizes and update budgets are reported. The repeated baseline is a reproducibility check, not another diffusion treatment. The completion marker is only written after all five checkpoints are complete.

In [ ]:
third_name = ('diffusion_real_only_control' if REPEAT_MODE == 'baseline_repeat'
              else 'filtered_plus_synthetic')
third_pool = (real_pool if REPEAT_MODE == 'baseline_repeat' else
              pd.concat([filtered_real_pool, synthetic_pool], ignore_index=True))
variants = [
    ('original_real_only', real_pool),
    ('original_overlap_rejected', filtered_real_pool),
    (third_name, third_pool),
    ('synthetic_only', synthetic_pool),
    ('real_plus_synthetic', augmented_pool),
]
EXPERIMENT_NAMES = ('original_real_only', 'original_overlap_rejected', third_name,
                    'synthetic_only', 'real_plus_synthetic')
EXPERIMENT_SEEDS = {name: SEED + 400 for name in EXPERIMENT_NAMES}
if REPEAT_MODE == 'baseline_repeat':
    EXPERIMENT_SEEDS[third_name] = SEED + 401
COMPLETION = SAVE_DIR / 'checkpoints' / 'all_experiments_complete.json'

def sha256_file(path):
    with path.open('rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()

def training_manifest_payload(data):
    return data[['image_path', 'label', 'target', 'source']].to_csv(index=False).encode('utf-8')

def verify_and_save_training_manifests():
    if sha256_file(TRAIN_MANIFEST) != train_split_sha:
        raise RuntimeError('Training split manifest changed.')
    if sha256_file(TEST_MANIFEST) != test_split_sha:
        raise RuntimeError('Held-out split manifest changed.')
    if train_pool.to_csv(index=False).encode('utf-8') != TRAIN_MANIFEST.read_bytes():
        raise RuntimeError('In-memory training pool differs from the frozen training split.')
    if synthetic_pool.to_csv(index=False).encode('utf-8') != (
            SAVE_DIR / 'results' / 'generated_manifest.csv').read_bytes():
        raise RuntimeError('Synthetic pool differs from the generated-image manifest.')
    expected = {
        'original_real_only': train_pool.assign(source='real'),
        'original_overlap_rejected': train_pool.loc[~rejected].reset_index(drop=True).assign(source='real'),
        third_name: (train_pool.assign(source='real') if REPEAT_MODE == 'baseline_repeat'
                     else pd.concat([train_pool.loc[~rejected].reset_index(drop=True).assign(source='real'),
                                     synthetic_pool], ignore_index=True)),
        'synthetic_only': synthetic_pool,
        'real_plus_synthetic': pd.concat(
            [train_pool.assign(source='real'), synthetic_pool], ignore_index=True)
    }
    manifest_dir = SAVE_DIR / 'splits' / 'experiments'
    manifest_dir.mkdir(exist_ok=True)
    fingerprints = {}
    for version, data in variants:
        assert_training_sources(data)
        payload = training_manifest_payload(data)
        if payload != training_manifest_payload(expected[version]):
            raise RuntimeError(f'Unexpected training data: {version}')
        if not data['image_path'].is_unique:
            raise RuntimeError(f'Duplicate training paths: {version}')
        manifest = manifest_dir / f'{version}.csv'
        if manifest.exists() and manifest.read_bytes() != payload:
            raise RuntimeError(f'Training manifest changed: {version}. Use a new SAVE_DIR.')
        if not manifest.exists():
            temp = manifest.with_suffix('.tmp')
            temp.write_bytes(payload)
            os.replace(temp, manifest)
        fingerprints[version] = sha256_file(manifest)
    return fingerprints

COMPLETION.unlink(missing_ok=True)
training_fingerprints = verify_and_save_training_manifests()
completed = []
for version, data in variants:
    name = f'final_resnet18_{version}'
    print(f'{name}: {len(data)} training images')
    model = train_model('resnet18', name, data, FINAL_EPOCHS, seed=EXPERIMENT_SEEDS[version])
    free_model(model)
    del model
    checkpoint = SAVE_DIR / 'checkpoints' / f'{name}.pt'
    saved = torch.load(checkpoint, map_location='cpu', weights_only=True)
    if saved['epoch'] != FINAL_EPOCHS:
        raise RuntimeError(f'Incomplete checkpoint: {name}')
    del saved
    completed.append({'data': version, 'checkpoint': checkpoint.name,
                      'sha256': sha256_file(checkpoint), 'epochs': FINAL_EPOCHS,
                      'seed': EXPERIMENT_SEEDS[version],
                      'training_manifest_sha256': training_fingerprints[version],
                      'train_images': len(data),
                      'optimizer_updates': FINAL_EPOCHS * math.ceil(len(data) / BATCH_SIZE)})
assert tuple(row['data'] for row in completed) == EXPERIMENT_NAMES
frozen = {'settings_sha256': sha256_file(SAVE_DIR / 'settings.json'),
          'train_split_sha256': train_split_sha, 'test_split_sha256': test_split_sha,
          'experiments': completed}
temp = COMPLETION.with_suffix('.tmp')
temp.write_text(json.dumps(frozen, indent=2))
os.replace(temp, COMPLETION)
print('All five training runs completed. No test evaluation performed.')

## 12. Final test evaluation — only after all five experiments finish

Verify the completion record, settings, fixed split manifests, training manifests and hashes of all five checkpoints before loading held-out test rows or images. Validate the disjointness of each training manifest from the test set. Reload frozen models in inference mode, use identical deterministic preprocessing and compute the same metrics for every model. No fitting, early stopping, model selection, or threshold adjustment takes place here.

Save test probabilities, per-image predictions, overall accuracy, balanced accuracy, macro precision, recall, F1, ROC-AUC and PR-AUC (average precision), per-class scores and evaluation provenance.

In [ ]:
def evaluate(y_true, probs):
    predicted = probs.argmax(axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, predicted, labels=np.arange(NUM_CLASSES),
        average='macro', zero_division=0
    )

    roc_scores = []
    pr_scores = []
    for label in range(NUM_CLASSES):
        is_class = (y_true == label).astype(int)
        if 0 < is_class.sum() < len(is_class):
            roc_scores.append(roc_auc_score(is_class, probs[:, label]))
            pr_scores.append(average_precision_score(is_class, probs[:, label]))

    return {
        'accuracy': accuracy_score(y_true, predicted),
        'balanced_accuracy': balanced_accuracy_score(y_true, predicted),
        'macro_precision': precision,
        'macro_recall': recall,
        'macro_f1': f1,
        'macro_roc_auc': np.mean(roc_scores) if roc_scores else np.nan,
        'macro_pr_auc': np.mean(pr_scores) if pr_scores else np.nan,
    }


def per_class_scores(y_true, probs, kind, version):
    predicted = probs.argmax(axis=1)
    p, r, f1, counts = precision_recall_fscore_support(
        y_true, predicted, labels=np.arange(NUM_CLASSES), zero_division=0
    )
    rows = []
    for label in range(NUM_CLASSES):
        binary = (y_true == label).astype(int)
        auc = roc_auc_score(binary, probs[:, label]) if 0 < binary.sum() < len(binary) else np.nan
        rows.append({'model': kind, 'data': version, 'class': class_names[label],
                     'support': counts[label], 'precision': p[label],
                     'recall': r[label], 'f1': f1[label], 'roc_auc': auc})
    return rows

In [ ]:
if not COMPLETION.is_file():
    raise RuntimeError('Complete all five training experiments before testing.')
frozen = json.loads(COMPLETION.read_text())
if (frozen['settings_sha256'] != sha256_file(SAVE_DIR / 'settings.json')
        or frozen['train_split_sha256'] != sha256_file(TRAIN_MANIFEST)
        or frozen['test_split_sha256'] != sha256_file(TEST_MANIFEST)):
    raise RuntimeError('Settings or split manifests changed.')
records = frozen['experiments']
if tuple(row['data'] for row in records) != EXPERIMENT_NAMES:
    raise RuntimeError('All five experiments must be complete.')
for row in records:
    manifest = SAVE_DIR / 'splits' / 'experiments' / f"{row['data']}.csv"
    if not manifest.is_file() or sha256_file(manifest) != row.get('training_manifest_sha256'):
        raise RuntimeError(f'Missing or changed training manifest: {manifest}. Rerun training section.')
    checkpoint = SAVE_DIR / 'checkpoints' / row['checkpoint']
    if not checkpoint.is_file() or sha256_file(checkpoint) != row['sha256']:
        raise RuntimeError(f'Missing or changed checkpoint: {checkpoint}')
    saved = torch.load(checkpoint, map_location='cpu', weights_only=True)
    if (saved['epoch'] != FINAL_EPOCHS or row['epochs'] != FINAL_EPOCHS
            or saved.get('signature', {}).get('seed') != row['seed']
            or row['seed'] != EXPERIMENT_SEEDS[row['data']]):
        raise RuntimeError(f'Incomplete checkpoint: {checkpoint}')
    del saved

test_data = pd.read_csv(TEST_MANIFEST, dtype={'label': 'string'})
test_paths = set(test_data['image_path'].map(lambda p: str((data_root / p).resolve())))
assert len(test_paths) == len(test_data)
assert test_data['target'].nunique() == NUM_CLASSES
assert (test_data['label'].map(class_to_id).astype(int) == test_data['target']).all()
for row in records:
    version = row['data']
    data = pd.read_csv(SAVE_DIR / 'splits' / 'experiments' / f'{version}.csv',
                       dtype={'label': 'string'})
    if len(data) != row['train_images']:
        raise RuntimeError(f'Training manifest size changed: {version}')
    training_paths = set(data['image_path'].map(lambda p: str((data_root / p).resolve())))
    if not training_paths.isdisjoint(test_paths):
        raise RuntimeError(f'Train/test path leakage: {version}')

test_labels = test_data['target'].to_numpy()
results, class_results = [], []
prediction_table = test_data[['image_path', 'label', 'target']].copy()
for row in records:
    version = row['data']
    name = f'final_resnet18_{version}'
    model = make_model('resnet18')
    saved = torch.load(SAVE_DIR / 'checkpoints' / row['checkpoint'],
                       map_location='cpu', weights_only=True)
    model.load_state_dict(saved['model'])
    del saved
    probs = predict_probs(model, test_data)
    assert probs.shape == (len(test_data), NUM_CLASSES)
    assert np.isfinite(probs).all() and np.allclose(probs.sum(axis=1), 1, atol=1e-4)
    np.save(SAVE_DIR / 'results' / f'{name}_test_probs.npy', probs)
    predicted = probs.argmax(axis=1)
    prediction_table[f'{version}_prediction'] = predicted
    prediction_table[f'{version}_predicted_label'] = [class_names[i] for i in predicted]
    prediction_table[f'{version}_correct'] = predicted == test_labels
    scores = evaluate(test_labels, probs)
    results.append({'model': 'resnet18', 'data': version,
                    'train_images': row['train_images'], 'test_images': len(test_data),
                    'optimizer_updates': row['optimizer_updates'], 'seed': row['seed'], **scores})
    class_results.extend(per_class_scores(test_labels, probs, 'resnet18', version))
    print(f"{version}: Test accuracy {scores['accuracy']:.4%}")
    free_model(model)
    del model

comparison = pd.DataFrame(results)
comparison.to_csv(SAVE_DIR / 'results' / 'model_comparison.csv', index=False)
pd.DataFrame(class_results).to_csv(SAVE_DIR / 'results' / 'per_class_metrics.csv', index=False)
prediction_table.to_csv(SAVE_DIR / 'results' / 'test_predictions.csv', index=False)
evaluation_record = {
    'settings_sha256': frozen['settings_sha256'],
    'test_split_sha256': frozen['test_split_sha256'],
    'test_images': len(test_data),
    'experiments': records
}
evaluation_path = SAVE_DIR / 'results' / 'test_evaluation_provenance.json'
temp = evaluation_path.with_suffix('.tmp')
temp.write_text(json.dumps(evaluation_record, indent=2))
os.replace(temp, evaluation_path)
print(comparison.round(4).to_string(index=False))

## 13. Compare all five test results

Report absolute scores and percentage-point differences against the original real-only baseline. Inspect the least frequent real classes as well as overall accuracy. These are newly computed test scores, not copied historical validation results.

In [ ]:
metrics = ['accuracy', 'balanced_accuracy', 'macro_precision', 'macro_recall',
           'macro_f1', 'macro_roc_auc', 'macro_pr_auc']
indexed = comparison.set_index('data')
changes = indexed.drop(index='original_real_only')[metrics].subtract(
    indexed.loc['original_real_only', metrics], axis='columns') * 100
changes.to_csv(SAVE_DIR / 'results' / 'all_experiments_minus_original_percentage_points.csv')
print('Change vs original real-only (percentage points):')
print(changes.round(3).to_string())

fig, axes = plt.subplots(1, 3, figsize=(19, 6))
for ax, metric in zip(axes, ['accuracy', 'balanced_accuracy', 'macro_f1']):
    ax.bar(comparison['data'], 100 * comparison[metric])
    ax.set_title(metric.replace('_', ' ').title())
    ax.set_ylabel('%')
    ax.set_ylim(0, 100)
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(SAVE_DIR / 'results' / 'five_experiment_test_comparison.png', dpi=150)
plt.show()

per_class = pd.DataFrame(class_results)
recalls = per_class.pivot(index='class', columns='data', values='recall')
recalls = recalls.join(class_counts.set_index('class')['real']).sort_values('real')
recalls['filtered_minus_original_pp'] = 100 * (recalls['original_overlap_rejected'] - recalls['original_real_only'])
recalls[f'{third_name}_minus_original_pp'] = 100 * (recalls[third_name] - recalls['original_real_only'])
recalls['augmented_minus_real_pp'] = 100 * (recalls['real_plus_synthetic'] - recalls['original_real_only'])
recalls['synthetic_minus_real_pp'] = 100 * (recalls['synthetic_only'] - recalls['original_real_only'])
recalls.to_csv(SAVE_DIR / 'results' / 'per_class_recall_comparison.csv')
print('20 least frequent real classes:')
print(recalls.head(20).round(4).to_string())
print('Results directory:', SAVE_DIR / 'results')

## 14. Interpretation and limitations

- All five final classifiers use one fixed test manifest, with no test images used during training, OOF rejection, or diffusion. No historical checkpoints or scores are reused.
- The baseline repeat uses the same data but a different seed. It is not an additional data-treatment condition. The optional filtered-plus-synthetic mode replaces that repeat with a fifth distinct treatment.
- The dataset was used in earlier experiments, so this is not a genuinely independent external benchmark. Path disjointness does not detect duplicate visual content at different paths.
- Generated labels are conditions rather than verified annotations. Inspect samples for fidelity, diversity and memorization. Class-balanced diffusion sampling can oversample rare real examples.
- Equal epochs are not compute-matched. Multiple seeds per condition and a compute-matched control are needed for stronger claims.
- Scores are generated only when this notebook is executed. Do not tune against the held-out test scores.